In [0]:
%sql
-- Criar view consolidada juntando as 4 tabelas
CREATE OR REPLACE TEMPORARY VIEW sales_master AS
SELECT 
    sp.opportunity_id,
    sp.sales_agent,
    sp.product,
    sp.account,
    sp.deal_stage,
    sp.engage_date,
    sp.close_date,
    sp.close_value,
    acc.sector,
    acc.revenue,
    acc.employees,
    acc.office_location,
    prod.series,
    prod.sales_price,
    team.manager,
    team.regional_office
FROM workspace.default.sales_pipeline sp
LEFT JOIN workspace.default.accounts acc ON sp.account = acc.account
LEFT JOIN workspace.default.products prod ON sp.product = prod.product
LEFT JOIN workspace.default.sales_teams team ON sp.sales_agent = team.sales_agent;

-- Checar volumetria por estágio
SELECT deal_stage, COUNT(*) AS total_deals
FROM sales_master
GROUP BY deal_stage
ORDER BY total_deals DESC;

deal_stage,total_deals
Won,4238
Lost,2473
Engaging,1589
Prospecting,500


In [0]:
%sql
-- Win Rate e Ticket Médio por Setor
SELECT 
    sector,
    COUNT(*) AS total_deals,
    SUM(CASE WHEN deal_stage = 'Won' THEN 1 ELSE 0 END) AS won_deals,
    ROUND(SUM(CASE WHEN deal_stage = 'Won' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS win_rate_pct,
    ROUND(AVG(close_value), 2) AS avg_close_value
FROM sales_master
WHERE deal_stage IN ('Won', 'Lost')
GROUP BY sector
ORDER BY win_rate_pct DESC;

sector,total_deals,won_deals,win_rate_pct,avg_close_value
marketing,623,404,64.85,1480.45
entertainment,402,260,64.68,1713.95
software,704,450,63.92,1531.16
technolgy,1058,671,63.42,1432.41
services,352,223,63.35,1514.22
retail,1267,799,63.06,1473.98
employment,286,179,62.59,1525.08
telecommunications,456,285,62.50,1433.28
medical,950,592,62.32,1431.15
finance,613,375,61.17,1551.24


In [0]:
%sql
-- Win Rate e Desempenho por Produto / Série
SELECT 
    series,
    product,
    sales_price,
    COUNT(*) AS total_deals,
    SUM(CASE WHEN deal_stage = 'Won' THEN 1 ELSE 0 END) AS won_deals,
    ROUND(SUM(CASE WHEN deal_stage = 'Won' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS win_rate_pct
FROM sales_master
WHERE deal_stage IN ('Won', 'Lost')
GROUP BY series, product, sales_price
ORDER BY win_rate_pct DESC;

series,product,sales_price,total_deals,won_deals,win_rate_pct
MG,MG Special,55,1223,793,64.84
GTX,GTX Plus Pro,5482,745,479,64.30
GTX,GTX Basic,550,1436,915,63.72
null,GTXPro,null,1147,729,63.56
GTX,GTX Plus Basic,1096,1051,653,62.13
MG,MG Advanced,3393,1084,654,60.33
GTK,GTK 500,26768,25,15,60.00


In [0]:
%sql
-- Win Rate por Escritório Regional e Manager
SELECT 
    regional_office,
    manager,
    COUNT(*) AS total_deals,
    SUM(CASE WHEN deal_stage = 'Won' THEN 1 ELSE 0 END) AS won_deals,
    ROUND(SUM(CASE WHEN deal_stage = 'Won' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS win_rate_pct
FROM sales_master
WHERE deal_stage IN ('Won', 'Lost')
GROUP BY regional_office, manager
ORDER BY win_rate_pct DESC;

regional_office,manager,total_deals,won_deals,win_rate_pct
East,Cara Losch,745,480,64.43
West,Summer Sewald,1287,828,64.34
West,Celia Rouche,962,610,63.41
Central,Dustin Brinkmann,1186,747,62.98
Central,Melvin Marxen,1418,882,62.20
East,Rocco Neubert,1113,691,62.08


In [0]:
%sql
-- Tempo Médio em Dias para Fechamento (Won vs Lost)
SELECT 
    deal_stage,
    COUNT(*) AS total_deals,
    ROUND(AVG(DATEDIFF(close_date, engage_date)), 1) AS avg_days_to_close,
    PERCENTILE(DATEDIFF(close_date, engage_date), 0.5) AS median_days_to_close,
    PERCENTILE(DATEDIFF(close_date, engage_date), 0.75) AS p75_days_to_close
FROM sales_master
WHERE deal_stage IN ('Won', 'Lost') AND engage_date IS NOT NULL
GROUP BY deal_stage;

deal_stage,total_deals,avg_days_to_close,median_days_to_close,p75_days_to_close
Won,4238,51.8,57.0,88.0
Lost,2473,41.5,14.0,79.0
